# Etapa 6 · Monitoreo de deriva de datos (*data drift*) con Evidently
**Proyecto Integrador — Heart Failure Prediction**

Un modelo se entrena con datos de un momento dado. Si más adelante llegan pacientes con
características distintas (más edad, más colesterol...), el modelo puede empeorar sin avisar.
A ese cambio se le llama **deriva de datos** (*data drift*).

Qué haremos:
1. Comparar los datos de **entrenamiento** (referencia) con los de **prueba** (actuales).
   Como vienen de la misma población, esperamos poca o ninguna deriva.
2. Simular una población distinta y ver cómo Evidently detecta la deriva.
3. Descargar el reporte HTML para tu entrega.

## 0. Preparación
Instalamos Evidently (versión fija, para que los resultados sean reproducibles) y subimos
`heart.csv` como en las etapas anteriores.

In [3]:
%pip install -q "evidently==0.7.23"

In [4]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from evidently import Report
from evidently.presets import DataDriftPreset

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

try:  # en Colab
    from google.colab import files
    subido = files.upload()
    RUTA = next(iter(subido))
except ImportError:  # fuera de Colab
    RUTA = "heart.csv"
print("Archivo:", RUTA)

Saving heart.csv to heart.csv
Archivo: heart.csv


## 1. Datos de referencia y datos actuales
Se repite la misma limpieza y la **misma división** de la Etapa 2 (misma semilla), así que
`X_train` y `X_test` son exactamente los conjuntos que ya conoces.

Para medir deriva usamos solo las **variables de entrada**, sin la respuesta.

In [5]:
df = pd.read_csv(RUTA, encoding="utf-8-sig")
TARGET = "HeartDisease" if "HeartDisease" in df.columns else "target"

for col in ["Cholesterol", "RestingBP"]:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

referencia = X_train.reset_index(drop=True)
actual = X_test.reset_index(drop=True)
print("Referencia (train):", referencia.shape, "| Actual (test):", actual.shape)

Referencia (train): (734, 11) | Actual (test): (184, 11)


## 2. Función de resumen
Evidently entrega un reporte HTML muy completo. Esta función además extrae lo esencial en
una tabla: para cada variable, el método estadístico usado, su valor y si hay deriva.

* Para variables numéricas usa la prueba de Kolmogorov-Smirnov; para categóricas, chi-cuadrado
  (o variantes). En esas pruebas hay deriva cuando el **p-valor es menor que 0.05**.
* Si usa una *distancia* (por ejemplo Jensen-Shannon), hay deriva cuando la distancia
  **supera el umbral**.

In [6]:
def resumen(snapshot):
    """Devuelve (cuántas variables derivaron, tabla por variable)."""
    filas, derivadas = [], None
    for m in snapshot.dict()["metrics"]:
        cfg = m["config"]
        tipo = cfg["type"]
        if tipo.endswith("DriftedColumnsCount"):
            derivadas = m["value"]
        elif tipo.endswith("ValueDrift"):
            metodo = str(cfg.get("method", ""))
            umbral = cfg.get("threshold")
            valor = m["value"]
            es_pvalor = "p_value" in metodo
            deriva = (valor < umbral) if es_pvalor else (valor >= umbral)
            filas.append({"variable": cfg["column"], "metodo": metodo,
                          "valor": round(float(valor), 4), "umbral": umbral,
                          "deriva": deriva})
    tabla = pd.DataFrame(filas).sort_values("deriva", ascending=False).reset_index(drop=True)
    tabla["deriva"] = tabla["deriva"].map({True: "SI", False: "no"})
    return derivadas, tabla

## 3. Escenario A: entrenamiento vs prueba (misma población)

In [7]:
reporte_A = Report([DataDriftPreset()])
snap_A = reporte_A.run(current_data=actual, reference_data=referencia)
snap_A.save_html("drift_report_A.html")

n, tabla_A = resumen(snap_A)
print(f"Variables con deriva: {int(n['count'])} de {len(tabla_A)} "
      f"({n['share']:.0%})")
tabla_A

Variables con deriva: 0 de 11 (0%)


,variable,metodo,valor,umbral,deriva
0,Age,K-S p_value,0.0721,0.05,no
1,RestingBP,K-S p_value,0.5770,0.05,no
2,Cholesterol,K-S p_value,0.1716,0.05,no
3,MaxHR,K-S p_value,0.6082,0.05,no
4,Oldpeak,K-S p_value,0.3229,0.05,no
5,Sex,Z-test p_value,0.8899,0.05,no
6,ChestPainType,chi-square p_value,0.1989,0.05,no
7,FastingBS,Z-test p_value,0.5447,0.05,no
8,RestingECG,chi-square p_value,0.5173,0.05,no
9,ExerciseAngina,Z-test p_value,0.5722,0.05,no


Con un nivel de 0.05 puede aparecer alguna variable marcada **por simple azar**: si miras 11
variables, es normal que una dé p < 0.05 sin que haya un cambio real. Por eso Evidently
declara "deriva del conjunto de datos" solo cuando una proporción grande de variables
(por defecto, la mitad) deriva.

## 4. Escenario B: una población distinta (deriva simulada)
Imagina que el modelo se despliega en una clínica que atiende pacientes **mayores**, con
**más colesterol** y **menor frecuencia cardíaca máxima**. Lo simulamos modificando los datos
actuales.

In [8]:
def col(*nombres):
    """Primera columna que exista entre los nombres dados."""
    return next(n for n in nombres if n in actual.columns)

c_edad = col("Age", "age")
c_chol = col("Cholesterol", "chol")
c_fc = col("MaxHR", "thalach")

actual_deriva = actual.copy()
actual_deriva[c_edad] = actual_deriva[c_edad] + 12
actual_deriva[c_chol] = actual_deriva[c_chol] * 1.30
actual_deriva[c_fc] = actual_deriva[c_fc] - 25

reporte_B = Report([DataDriftPreset()])
snap_B = reporte_B.run(current_data=actual_deriva, reference_data=referencia)
snap_B.save_html("drift_report_B.html")

n, tabla_B = resumen(snap_B)
print(f"Variables con deriva: {int(n['count'])} de {len(tabla_B)} "
      f"({n['share']:.0%})")
tabla_B

Variables con deriva: 3 de 11 (27%)


,variable,metodo,valor,umbral,deriva
0,Age,K-S p_value,0.0000,0.05,SI
1,Cholesterol,K-S p_value,0.0000,0.05,SI
2,MaxHR,K-S p_value,0.0000,0.05,SI
3,RestingBP,K-S p_value,0.5770,0.05,no
4,Oldpeak,K-S p_value,0.3229,0.05,no
5,Sex,Z-test p_value,0.8899,0.05,no
6,ChestPainType,chi-square p_value,0.1989,0.05,no
7,FastingBS,Z-test p_value,0.5447,0.05,no
8,RestingECG,chi-square p_value,0.5173,0.05,no
9,ExerciseAngina,Z-test p_value,0.5722,0.05,no


**Cómo leerlo.** Las tres variables que modificamos deben aparecer con `deriva = SI`. Evidently
también marca otras variables si el cambio las afecta, pero no cambiamos nada más.
Cuando en producción vea algo así, la acción razonable es **revisar el modelo y, si hace falta,
reentrenarlo** con datos recientes.

## 5. Descargar los reportes
Los `.html` son interactivos: ábrelos en tu navegador para ver, variable por variable, la
distribución de referencia contra la actual.

In [9]:
try:
    files.download("drift_report_A.html")
    files.download("drift_report_B.html")
except NameError:
    print("Fuera de Colab: los archivos quedaron en la carpeta actual.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Para cerrar la Etapa 6
Escribe en 3–4 líneas:
* qué resultado dio el escenario A (¿hubo deriva en el conjunto?) y por qué se esperaba;
* qué variables detectó Evidently en el escenario B;
* qué harías en producción si vieras un reporte como el B.